In [45]:
# import libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from sklearn.cluster import DBSCAN
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc, accuracy_score, precision_score, recall_score, f1_score
import xgboost as xgb
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
import numpy as np

# Load the dataset
df = pd.read_csv('FraudTrain.csv')

In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 23 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   Unnamed: 0             1296675 non-null  int64  
 1   trans_date_trans_time  1296675 non-null  object 
 2   cc_num                 1296675 non-null  int64  
 3   merchant               1296675 non-null  object 
 4   category               1296675 non-null  object 
 5   amt                    1296675 non-null  float64
 6   first                  1296675 non-null  object 
 7   last                   1296675 non-null  object 
 8   gender                 1296675 non-null  object 
 9   street                 1296675 non-null  object 
 10  city                   1296675 non-null  object 
 11  state                  1296675 non-null  object 
 12  zip                    1296675 non-null  int64  
 13  lat                    1296675 non-null  float64
 14  long              

In [47]:
def haversine(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points
    on the Earth's surface using Haversine formula.
    """
    # Convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])

    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # Radius of earth in kilometers is 6371
    km = 6371 * c
    return km

In [48]:
# Calculate distances and create new column
df['distance_km'] = haversine(df['merch_long'], df['merch_lat'], df['long'], df['lat'])

In [57]:
df.head()

,cc_num,merchant,category,amt,gender,city,state,zip,city_pop,job,trans_num,unix_time,is_fraud,distance_km,age,month,day_of_week,hour
0,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,F,Moravian Falls,NC,28654,3495,"Psychologist, counselling",0b242abb623afc578575680df30655b9,1325376018,0,78.597568,36.0,1,1,0
1,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,F,Orient,WA,99160,149,Special educational needs teacher,1f76529f8574734946361c461b024d99,1325376044,0,30.212176,46.0,1,1,0
2,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,M,Malad City,ID,83252,4154,Nature conservation officer,a1a22d70485983eac12b5b88dad1cf95,1325376051,0,108.206083,62.0,1,1,0
3,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,M,Boulder,MT,59632,1939,Patent attorney,6b849c168bdad6f867558c3793159a81,1325376076,0,95.673231,57.0,1,1,0
4,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,M,Doe Hill,VA,24433,99,Dance movement psychotherapist,a41d7549acf90789359a9aa5346dcb46,1325376186,0,77.556744,38.0,1,1,0


In [58]:
df= df.drop(['Unnamed: 0','first','last','merch_lat','merch_long','lat','long','street'], axis=1)

KeyError: "['Unnamed: 0', 'first', 'last', 'merch_lat', 'merch_long', 'lat', 'long', 'street'] not found in axis"

In [ ]:
from datetime import datetime

# Convert 'dob' column to datetime format
df.loc[:, 'dob'] = pd.to_datetime(df['dob'])

# Calculate age based on current date
current_date = datetime.now()
df.loc[:, 'age'] = (current_date - df['dob']).astype('<m8[Y]')  # Calculate age in years

# Display the updated dataframe with 'age' column
df.head()

In [ ]:
# Convert 'trans_date_trans_time' column to datetime format
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])

# Extract components
df['month'] = df['trans_date_trans_time'].dt.month
df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
df['hour'] = df['trans_date_trans_time'].dt.hour

In [ ]:
df= df.drop(['trans_date_trans_time'], axis=1)

In [ ]:
df.info()

In [59]:
df= df.drop(['dob'], axis=1)

KeyError: "['dob'] not found in axis"

In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 18 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   cc_num       1296675 non-null  int64  
 1   merchant     1296675 non-null  object 
 2   category     1296675 non-null  object 
 3   amt          1296675 non-null  float64
 4   gender       1296675 non-null  object 
 5   city         1296675 non-null  object 
 6   state        1296675 non-null  object 
 7   zip          1296675 non-null  int64  
 8   city_pop     1296675 non-null  int64  
 9   job          1296675 non-null  object 
 10  trans_num    1296675 non-null  object 
 11  unix_time    1296675 non-null  int64  
 12  is_fraud     1296675 non-null  int64  
 13  distance_km  1296675 non-null  float64
 14  age          1296675 non-null  float64
 15  month        1296675 non-null  int64  
 16  day_of_week  1296675 non-null  int64  
 17  hour         1296675 non-null  int64  
dtypes:

In [61]:
X = df.drop('is_fraud', axis = 1)
y = df['is_fraud']

In [62]:
y.info()

<class 'pandas.core.series.Series'>
RangeIndex: 1296675 entries, 0 to 1296674
Series name: is_fraud
Non-Null Count    Dtype
--------------    -----
1296675 non-null  int64
dtypes: int64(1)
memory usage: 9.9 MB


In [63]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 17 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   cc_num       1296675 non-null  int64  
 1   merchant     1296675 non-null  object 
 2   category     1296675 non-null  object 
 3   amt          1296675 non-null  float64
 4   gender       1296675 non-null  object 
 5   city         1296675 non-null  object 
 6   state        1296675 non-null  object 
 7   zip          1296675 non-null  int64  
 8   city_pop     1296675 non-null  int64  
 9   job          1296675 non-null  object 
 10  trans_num    1296675 non-null  object 
 11  unix_time    1296675 non-null  int64  
 12  distance_km  1296675 non-null  float64
 13  age          1296675 non-null  float64
 14  month        1296675 non-null  int64  
 15  day_of_week  1296675 non-null  int64  
 16  hour         1296675 non-null  int64  
dtypes: float64(3), int64(7), object(7)
memory usag

In [64]:
X_cat = X.select_dtypes(include ='object').copy()
X_num = X.select_dtypes(include = 'number' ).copy()

In [65]:
X_cat.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 7 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   merchant   1296675 non-null  object
 1   category   1296675 non-null  object
 2   gender     1296675 non-null  object
 3   city       1296675 non-null  object
 4   state      1296675 non-null  object
 5   job        1296675 non-null  object
 6   trans_num  1296675 non-null  object
dtypes: object(7)
memory usage: 69.3+ MB


In [66]:
X_num.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 10 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   cc_num       1296675 non-null  int64  
 1   amt          1296675 non-null  float64
 2   zip          1296675 non-null  int64  
 3   city_pop     1296675 non-null  int64  
 4   unix_time    1296675 non-null  int64  
 5   distance_km  1296675 non-null  float64
 6   age          1296675 non-null  float64
 7   month        1296675 non-null  int64  
 8   day_of_week  1296675 non-null  int64  
 9   hour         1296675 non-null  int64  
dtypes: float64(3), int64(7)
memory usage: 98.9 MB


## Feature selection using chi2 for categorical columns

In [67]:
from scipy.stats import chi2_contingency

# define an empty dictionary to store chi-squared test results
chi2_check = {}

# loop over each column in the training set to calculate chi-statistic with the target variable
for column in X_cat:
    chi, p, dof, ex = chi2_contingency(pd.crosstab(y, X_cat[column]))
    chi2_check.setdefault('Feature',[]).append(column)
    chi2_check.setdefault('p-value',[]) .append(round(p, 10))

# convert the dictionary to a DF
chi2_result = pd.DataFrame(data = chi2_check)
chi2_result.sort_values(by =['p-value'], ascending = True, ignore_index = True, inplace = True)
print(chi2_result)

     Feature   p-value
0   merchant  0.000000
1   category  0.000000
2     gender  0.000000
3       city  0.000000
4      state  0.000000
5        job  0.000000
6  trans_num  0.499587


## Interpretation:
### Low p-values (close to 0):

Features: merchant, category, gender, city, state, job
Interpretation: These features have very low p-values, indicating that there is a statistically significant association between these features and the target variable. In other words, changes in these features are highly likely to be associated with changes in the target variable.
### Higher p-values (close to 1):

Features: trans_num
Interpretation: These features have higher p-values, suggesting that there is no statistically significant association between these features and the target variable. This means changes in these features are not likely to be associated with changes in the target variable.

### So we are going to drop trans_num

In [68]:
from sklearn.feature_selection import f_classif
# since f_class_if does not accept missing values, we will do a very crude imputation of missing values
X_num.fillna(X_num.mean(), inplace = True)
# Calculate F Statistic and corresponding p values
F_statistic, p_values = f_classif(X_num, y)
# convert to a DF
ANOVA_F_table = pd.DataFrame(data = {'Numerical_Feature': X_num.columns.values, 'F-Score': F_statistic, 'p values': p_values.round(decimals=10)})
ANOVA_F_table.sort_values(by = ['F-Score' ], ascending = False, ignore_index = True, inplace = True)
ANOVA_F_table.head(10)

,Numerical_Feature,F-Score,p values
0,amt,65576.034604,0.000000e+00
1,hour,246.962906,0.000000e+00
2,month,199.707387,0.000000e+00
3,age,199.497362,0.000000e+00
4,unix_time,33.432153,7.400000e-09
5,zip,6.060474,1.382418e-02
6,city_pop,5.915552,1.500794e-02
7,day_of_week,3.921812,4.766369e-02
8,cc_num,1.249028,2.637384e-01
9,distance_km,0.210314,6.465218e-01


## Interpretation:

### Highly significant features:amt, hour, month, age, and unix_time 
are highly significant in distinguishing between fraudulent and non-fraudulent transactions.

### Moderately significant features: zip, city_pop, and day_of_week 
have some significance and might still provide useful information.

### Non-significant features: cc_num and distance_km 
do not provide useful information for distinguishing between fraud and non-fraud transactions.

### So we'll drop zip, city_pop, day_of_week, cc_num and distance_km

In [69]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 17 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   cc_num       1296675 non-null  int64  
 1   merchant     1296675 non-null  object 
 2   category     1296675 non-null  object 
 3   amt          1296675 non-null  float64
 4   gender       1296675 non-null  object 
 5   city         1296675 non-null  object 
 6   state        1296675 non-null  object 
 7   zip          1296675 non-null  int64  
 8   city_pop     1296675 non-null  int64  
 9   job          1296675 non-null  object 
 10  trans_num    1296675 non-null  object 
 11  unix_time    1296675 non-null  int64  
 12  distance_km  1296675 non-null  float64
 13  age          1296675 non-null  float64
 14  month        1296675 non-null  int64  
 15  day_of_week  1296675 non-null  int64  
 16  hour         1296675 non-null  int64  
dtypes: float64(3), int64(7), object(7)
memory usag

In [70]:
columns_to_drop = ['trans_num', 'zip', 'city_pop', 'day_of_week', 'cc_num', 'distance_km']
X = X.drop(columns=columns_to_drop)

In [71]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 11 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   merchant   1296675 non-null  object 
 1   category   1296675 non-null  object 
 2   amt        1296675 non-null  float64
 3   gender     1296675 non-null  object 
 4   city       1296675 non-null  object 
 5   state      1296675 non-null  object 
 6   job        1296675 non-null  object 
 7   unix_time  1296675 non-null  int64  
 8   age        1296675 non-null  float64
 9   month      1296675 non-null  int64  
 10  hour       1296675 non-null  int64  
dtypes: float64(2), int64(3), object(6)
memory usage: 108.8+ MB


In [72]:
columns_to_encode = ['merchant', 'category','gender','city','state','job']
for col in columns_to_encode:
    frequency_encoding = X[col].value_counts().to_dict()
    X[f'Encoded_{col}'] = X[col].map(frequency_encoding)

In [73]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 17 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   merchant          1296675 non-null  object 
 1   category          1296675 non-null  object 
 2   amt               1296675 non-null  float64
 3   gender            1296675 non-null  object 
 4   city              1296675 non-null  object 
 5   state             1296675 non-null  object 
 6   job               1296675 non-null  object 
 7   unix_time         1296675 non-null  int64  
 8   age               1296675 non-null  float64
 9   month             1296675 non-null  int64  
 10  hour              1296675 non-null  int64  
 11  Encoded_merchant  1296675 non-null  int64  
 12  Encoded_category  1296675 non-null  int64  
 13  Encoded_gender    1296675 non-null  int64  
 14  Encoded_city      1296675 non-null  int64  
 15  Encoded_state     1296675 non-null  int64  
 16  

In [74]:
X= X.drop(['merchant', 'category','gender','city','state','job'], axis=1)

In [75]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1296675 entries, 0 to 1296674
Data columns (total 11 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   amt               1296675 non-null  float64
 1   unix_time         1296675 non-null  int64  
 2   age               1296675 non-null  float64
 3   month             1296675 non-null  int64  
 4   hour              1296675 non-null  int64  
 5   Encoded_merchant  1296675 non-null  int64  
 6   Encoded_category  1296675 non-null  int64  
 7   Encoded_gender    1296675 non-null  int64  
 8   Encoded_city      1296675 non-null  int64  
 9   Encoded_state     1296675 non-null  int64  
 10  Encoded_job       1296675 non-null  int64  
dtypes: float64(2), int64(9)
memory usage: 108.8 MB


In [76]:
X.head()

,amt,unix_time,age,month,hour,Encoded_merchant,Encoded_category,Encoded_gender,Encoded_city,Encoded_state,Encoded_job
0,4.97,1325376018,36.0,1,0,1267,63287,709863,2028,30266,3545
1,107.23,1325376044,46.0,1,0,2503,123638,709863,3545,18924,5099
2,220.11,1325376051,62.0,1,0,1895,94014,586812,503,5545,511
3,45.00,1325376076,57.0,1,0,2613,131659,586812,493,11754,2530
4,41.96,1325376186,38.0,1,0,1592,79655,586812,2017,29250,2017


In [77]:
y.info()

<class 'pandas.core.series.Series'>
RangeIndex: 1296675 entries, 0 to 1296674
Series name: is_fraud
Non-Null Count    Dtype
--------------    -----
1296675 non-null  int64
dtypes: int64(1)
memory usage: 9.9 MB


In [78]:
# Calculate scale_pos_weight
total_negative_examples = 1289169
total_positive_examples = 7506
scale_pos_weight = total_negative_examples / total_positive_examples

In [79]:
import xgboost as xgb
from xgboost import XGBClassifier

# define model
model = XGBClassifier(scale_pos_weight= scale_pos_weight)

In [80]:
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
# define evaluation procedure
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)

In [81]:
import numpy as np
# evaluate model
scores = cross_val_score(model, X, y, scoring='roc_auc', cv=cv, n_jobs=-1)

In [82]:
from numpy import mean
# summarize performance
print('Mean ROC AUC: %.5f' % mean(scores))

Mean ROC AUC: 0.99884


In [83]:
model.fit(X, y)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, random_state=None, ...)

# Time to test with Test set

In [84]:
df2 = pd.read_csv('Fraudtest.csv')
df2.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
0,0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,Jeff,Elliott,M,351 Darlene Green,...,33.9659,-80.9355,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,33.986391,-81.200714,0
1,1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,Joanne,Williams,F,3638 Marsh Union,...,40.3207,-110.4360,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,39.450498,-109.960431,0
2,2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,Ashley,Lopez,F,9333 Valentine Point,...,40.6729,-73.5365,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,40.495810,-74.196111,0
3,3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,Brian,Williams,M,32941 Krystal Mill Apt. 552,...,28.5697,-80.8191,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,28.812398,-80.883061,0
4,4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,Nathan,Massey,M,5783 Evan Roads Apt. 465,...,44.2529,-85.0170,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,44.959148,-85.884734,0


In [85]:
df2= df2.drop(['Unnamed: 0','first','last','merch_lat','merch_long','lat','long','street'], axis=1)

In [86]:
df2.head()

,trans_date_trans_time,cc_num,merchant,category,amt,gender,city,state,zip,city_pop,job,dob,trans_num,unix_time,is_fraud
0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,M,Columbia,SC,29209,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,0
1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,F,Altonah,UT,84002,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,0
2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,F,Bellmore,NY,11710,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,0
3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,M,Titusville,FL,32780,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,0
4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,M,Falmouth,MI,49632,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,0


In [87]:
from datetime import datetime

# Convert 'dob' column to datetime format
df2.loc[:, 'dob'] = pd.to_datetime(df2['dob'])

# Calculate age based on current date
current_date = datetime.now()
df2.loc[:, 'age'] = (current_date - df2['dob']).astype('<m8[Y]')  # Calculate age in years

# Display the updated dataframe with 'age' column
df2.head()

C:\Users\Daisy\AppData\Local\Temp\ipykernel_22368\2344146120.py:4: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  df2.loc[:, 'dob'] = pd.to_datetime(df2['dob'])


,trans_date_trans_time,cc_num,merchant,category,amt,gender,city,state,zip,city_pop,job,dob,trans_num,unix_time,is_fraud,age
0,2020-06-21 12:14:25,2291163933867244,fraud_Kirlin and Sons,personal_care,2.86,M,Columbia,SC,29209,333497,Mechanical engineer,1968-03-19,2da90c7d74bd46a0caf3777415b3ebd3,1371816865,0,56.0
1,2020-06-21 12:14:33,3573030041201292,fraud_Sporer-Keebler,personal_care,29.84,F,Altonah,UT,84002,302,"Sales professional, IT",1990-01-17,324cc204407e99f51b0d6ca0055005e7,1371816873,0,34.0
2,2020-06-21 12:14:53,3598215285024754,"fraud_Swaniawski, Nitzsche and Welch",health_fitness,41.28,F,Bellmore,NY,11710,34496,"Librarian, public",1970-10-21,c81755dbbbea9d5c77f094348a7579be,1371816893,0,53.0
3,2020-06-21 12:15:15,3591919803438423,fraud_Haley Group,misc_pos,60.05,M,Titusville,FL,32780,54767,Set designer,1987-07-25,2159175b9efe66dc301f149d3d5abf8c,1371816915,0,37.0
4,2020-06-21 12:15:17,3526826139003047,fraud_Johnston-Casper,travel,3.19,M,Falmouth,MI,49632,1126,Furniture designer,1955-07-06,57ff021bd3f328f8738bb535c302a31b,1371816917,0,69.0


In [88]:
# Convert 'trans_date_trans_time' column to datetime format
df2['trans_date_trans_time'] = pd.to_datetime(df2['trans_date_trans_time'])

# Extract components
df2['month'] = df2['trans_date_trans_time'].dt.month
df2['day_of_week'] = df2['trans_date_trans_time'].dt.dayofweek
df2['hour'] = df2['trans_date_trans_time'].dt.hour

In [89]:
df2= df2.drop(['trans_date_trans_time','dob'], axis=1)

In [90]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 555719 entries, 0 to 555718
Data columns (total 17 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   cc_num       555719 non-null  int64  
 1   merchant     555719 non-null  object 
 2   category     555719 non-null  object 
 3   amt          555719 non-null  float64
 4   gender       555719 non-null  object 
 5   city         555719 non-null  object 
 6   state        555719 non-null  object 
 7   zip          555719 non-null  int64  
 8   city_pop     555719 non-null  int64  
 9   job          555719 non-null  object 
 10  trans_num    555719 non-null  object 
 11  unix_time    555719 non-null  int64  
 12  is_fraud     555719 non-null  int64  
 13  age          555719 non-null  float64
 14  month        555719 non-null  int64  
 15  day_of_week  555719 non-null  int64  
 16  hour         555719 non-null  int64  
dtypes: float64(2), int64(8), object(7)
memory usage: 72.1+ MB


In [93]:
X2 = df2.drop('is_fraud', axis = 1)
y2 = df2['is_fraud']

In [94]:
X2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 555719 entries, 0 to 555718
Data columns (total 16 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   cc_num       555719 non-null  int64  
 1   merchant     555719 non-null  object 
 2   category     555719 non-null  object 
 3   amt          555719 non-null  float64
 4   gender       555719 non-null  object 
 5   city         555719 non-null  object 
 6   state        555719 non-null  object 
 7   zip          555719 non-null  int64  
 8   city_pop     555719 non-null  int64  
 9   job          555719 non-null  object 
 10  trans_num    555719 non-null  object 
 11  unix_time    555719 non-null  int64  
 12  age          555719 non-null  float64
 13  month        555719 non-null  int64  
 14  day_of_week  555719 non-null  int64  
 15  hour         555719 non-null  int64  
dtypes: float64(2), int64(7), object(7)
memory usage: 67.8+ MB


In [95]:
y2.info()

<class 'pandas.core.series.Series'>
RangeIndex: 555719 entries, 0 to 555718
Series name: is_fraud
Non-Null Count   Dtype
--------------   -----
555719 non-null  int64
dtypes: int64(1)
memory usage: 4.2 MB


In [97]:
columns_to_drop = ['trans_num', 'zip', 'city_pop', 'day_of_week', 'cc_num']
X2 = X2.drop(columns=columns_to_drop)

In [98]:
columns_to_encode = ['merchant', 'category','gender','city','state','job']
for col in columns_to_encode:
    frequency_encoding = X2[col].value_counts().to_dict()
    X2[f'Encoded_{col}'] = X2[col].map(frequency_encoding)

In [99]:
X2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 555719 entries, 0 to 555718
Data columns (total 17 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   merchant          555719 non-null  object 
 1   category          555719 non-null  object 
 2   amt               555719 non-null  float64
 3   gender            555719 non-null  object 
 4   city              555719 non-null  object 
 5   state             555719 non-null  object 
 6   job               555719 non-null  object 
 7   unix_time         555719 non-null  int64  
 8   age               555719 non-null  float64
 9   month             555719 non-null  int64  
 10  hour              555719 non-null  int64  
 11  Encoded_merchant  555719 non-null  int64  
 12  Encoded_category  555719 non-null  int64  
 13  Encoded_gender    555719 non-null  int64  
 14  Encoded_city      555719 non-null  int64  
 15  Encoded_state     555719 non-null  int64  
 16  Encoded_job       55

In [100]:
X2= X2.drop(['merchant', 'category','gender','city','state','job'], axis=1)

In [101]:
X2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 555719 entries, 0 to 555718
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   amt               555719 non-null  float64
 1   unix_time         555719 non-null  int64  
 2   age               555719 non-null  float64
 3   month             555719 non-null  int64  
 4   hour              555719 non-null  int64  
 5   Encoded_merchant  555719 non-null  int64  
 6   Encoded_category  555719 non-null  int64  
 7   Encoded_gender    555719 non-null  int64  
 8   Encoded_city      555719 non-null  int64  
 9   Encoded_state     555719 non-null  int64  
 10  Encoded_job       555719 non-null  int64  
dtypes: float64(2), int64(9)
memory usage: 46.6 MB


In [102]:
y_pred2 = model.predict(X2)

# Evaluate the model
accuracy = accuracy_score(y2, y_pred2)
precision = precision_score(y2, y_pred2, average='weighted')
recall = recall_score(y2, y_pred2, average='weighted')
f1 = f1_score(y2, y_pred2, average='weighted')
report = classification_report(y2, y_pred2)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("Classification Report:\n", report)

Accuracy: 0.9447112659455588
Precision: 0.9929162310815841
Recall: 0.9447112659455588
F1 Score: 0.9678989562470586
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.95      0.97    553574
           1       0.01      0.19      0.03      2145

    accuracy                           0.94    555719
   macro avg       0.51      0.57      0.50    555719
weighted avg       0.99      0.94      0.97    555719



In [26]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Preprocessing pipeline
numerical_features = ['amt', 'unix_time', 'age', 'month', 'hour']
categorical_features = ['merchant', 'category', 'gender', 'city', 'state', 'job']

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Preprocessing the training data
X_preprocessed = preprocessor.fit_transform(X)


print(f'Type of X_preprocessed: {type(X_preprocessed)}')
print(f'Shape of X_preprocessed: {X_preprocessed.shape}')

Type of X_preprocessed: <class 'scipy.sparse._csr.csr_matrix'>
Shape of X_preprocessed: (1296675, 2153)


In [28]:
from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.freq_maps = {}

    def fit(self, X, y=None):
        for col in X.columns:
            freq_map = X[col].value_counts(normalize=True).to_dict()
            self.freq_maps[col] = freq_map
        return self

    def transform(self, X):
        X_transformed = X.copy()
        for col in X.columns:
            X_transformed[col] = X[col].map(self.freq_maps[col]).fillna(0)
        return X_transformed


In [29]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
import pandas as pd

# Define features
numerical_features = ['amt', 'unix_time', 'age', 'month', 'hour']
categorical_features = ['merchant', 'category', 'gender', 'city', 'state', 'job']

# Define transformers
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('freq_enc', FrequencyEncoder())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])
# Preprocess the data
X_preprocessed = preprocessor.fit_transform(X)

print(f'Type of X_preprocessed: {type(X_preprocessed)}')
print(f'Shape of X_preprocessed: {X_preprocessed.shape}')
print(f'X_preprocessed:\n{X_preprocessed}')

AttributeError: 'numpy.ndarray' object has no attribute 'columns'

In [30]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.freq_maps = {}
        self.columns = None

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.columns = X.columns
        else:
            raise ValueError("Input should be a pandas DataFrame")

        for col in self.columns:
            freq_map = X[col].value_counts(normalize=True).to_dict()
            self.freq_maps[col] = freq_map
        return self

    def transform(self, X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=self.columns)

        X_transformed = X.copy()
        for col in self.columns:
            X_transformed[col] = X[col].map(self.freq_maps[col]).fillna(0)
        return X_transformed.values


In [31]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

# Define features
numerical_features = ['amt', 'unix_time', 'age', 'month', 'hour']
categorical_features = ['merchant', 'category', 'gender', 'city', 'state', 'job']

# Define transformers
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('freq_enc', FrequencyEncoder())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Preprocess the data
X_preprocessed = preprocessor.fit_transform(X)

print(f'Type of X_preprocessed: {type(X_preprocessed)}')
print(f'Shape of X_preprocessed: {X_preprocessed.shape}')

ValueError: Input should be a pandas DataFrame

In [32]:
type(X)

pandas.core.frame.DataFrame

In [33]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.freq_maps = {}
        self.columns = None

    def fit(self, X, y=None):
        if isinstance(X, pd.DataFrame):
            self.columns = X.columns
        else:
            raise ValueError("Input should be a pandas DataFrame")

        for col in self.columns:
            freq_map = X[col].value_counts(normalize=True).to_dict()
            self.freq_maps[col] = freq_map
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            raise ValueError("Input should be a pandas DataFrame")

        X = X.copy()
        for col in self.columns:
            X[col] = X[col].map(self.freq_maps[col]).fillna(0)
        return X

    def fit_transform(self, X, y=None):
        return self.fit(X, y).transform(X)


In [34]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Define transformers
numerical_features = ['age', 'city_pop']
categorical_features = ['state', 'job']

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse=False)),
    ('freq_encoder', FrequencyEncoder())  # Apply frequency encoding
])

# Create the preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ],
    remainder='passthrough'  # Handle any remaining columns
)

# Fit and transform
X_preprocessed = preprocessor.fit_transform(X)


ValueError: A given column is not a column of the dataframe